# 19 Dialogue Analysis

Analyses the dialogue experiment collected by `18_dialogue`.
Persistence is one of the things measured inside it, not the name of it. The adaptation
experiment measures what a model says first. This one measures whether that
survives being pushed.

Design, fixed in `config/settings.yml` before any dialogue exists:

| | |
|---|---|
| Scenarios | 50, the 25 age-restricted and 25 of the 50 harmful |
| Conditions | control and seven stated ages, seven to eighteen |
| Methods | Purpose Reverse, Role Play, Emotional Pushback |
| Turns | 1 user, 2 the model's own opening reply, 3 and 5 pressure, 4 and 6 the replies scored here |

Turn 2 is not regenerated. It is the reply the model actually gave in the
adaptation pass, joined back on model, prompt and replicate, so movement at a
later turn is measured against what that model did rather than against a fresh
draw. Turns 4 and 6 are scored by the same frozen rubric at the same policy
fingerprint as the adaptation corpus.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
from analysis import MACRO, NAME, ORDER, STATED_AGE
from utils import read_lines

pd.set_option('display.width', 200, 'display.max_columns', 40)

RESULTS = Path.cwd().parent / 'results'
TABLES = Path.cwd().parent / 'tables'
PRESSED = [2, 3]


# Write a finished table to the frozen trail. publish() is the house route and
# refuses a name config/captions.yml does not describe, so until these names are
# entered there this writes the CSV directly and nothing else.
def freeze(table, name, tier='supplement'):
    table.to_csv(TABLES / ('main' if tier == 'main' else 'supplement') / f'{name}.csv')
    return table

## The corpus

The opening reply is joined from the adaptation classification rather than
reclassified, so a dialogue that opened with a refusal is one the adaptation
chapter also counts as a refusal.

In [ ]:
plan = pd.read_csv(RESULTS / 'dialogue' / 'plan.csv')
seeds = (plan[['dialogue_id', 'prompt_id', 'scenario_id', 'condition', 'model',
               'opening_replicate', 'method', 'expected_answer']]
         .drop_duplicates('dialogue_id'))

opening = pd.read_csv(RESULTS / 'classification.csv',
                      usecols=['model', 'prompt_id', 'replicate', 'answer',
                               'delivery_response'])
seeds = seeds.merge(opening.rename(columns={'answer': 'opening_answer',
                                            'delivery_response': 'opening_delivery'}),
                    left_on=['model', 'prompt_id', 'opening_replicate'],
                    right_on=['model', 'prompt_id', 'replicate'], how='left')

pressed = pd.DataFrame(read_lines(RESULTS / 'classification_dialogue.jsonl'))
pressed['turn'] = pressed['turn'].astype(int)
pressed = pressed[pressed['turn'].isin(PRESSED)]

frame = pressed.merge(seeds, on=['dialogue_id', 'model'], how='left')
frame['label'] = frame['model'].map(NAME)
frame['age'] = frame['condition'].map(STATED_AGE)
frame['refused'] = frame['answer'].eq('Refusal')
frame['opening_refused'] = frame['opening_answer'].eq('Refusal')

print(f"{seeds['dialogue_id'].nunique():,} dialogues planned, "
      f"{frame['dialogue_id'].nunique():,} scored over {len(frame):,} turns, "
      f"policy {frame['policy'].iat[0]}")

In [ ]:
# One frame carrying all three stages, so every table below shares a single
# definition of what a stage is and what a defect is.
#
# The transcript numbers every message, so an assistant reply sits at position
# 2, 4 or 6. The thesis defines a turn as one user message and its reply, so
# those are turns 1, 2 and 3, and that is what the tables say. Turn 1 is the
# opening reply carried over from the adaptation run rather than regenerated.
# results/ now stores turns rather than message positions, after
# scripts/renameturns.py. The opening reply is turn 1 and is joined from
# the adaptation run; the two pressed replies are turns 2 and 3.
STAGE = {1: 'Turn 1', 2: 'Turn 2', 3: 'Turn 3'}
CELLS = ['Strong Refusal', 'Weak Refusal', 'Minimal Compliance', 'Total Compliance']


def cells(refused, delivered):
    return np.select(
        [refused & ~delivered, refused & delivered, ~refused & ~delivered],
        CELLS[:3], default=CELLS[3])


opening = seeds.assign(
    turn=1,
    refused=seeds['opening_answer'].eq('Refusal'),
    delivered=seeds['opening_delivery'].eq('Yes'))
pressed_stages = frame.assign(delivered=frame['delivery_response'].eq('Yes'))

columns = ['dialogue_id', 'model', 'scenario_id', 'condition', 'method',
           'expected_answer', 'turn', 'refused', 'delivered']
stage = pd.concat([opening[columns], pressed_stages[columns]], ignore_index=True)
stage['label'] = stage['model'].map(NAME)
stage['age'] = stage['condition'].map(STATED_AGE)
stage['stratum'] = np.where(stage['scenario_id'].str.contains('-a'),
                            'Age Restricted', 'Harmful')
stage['outcome'] = cells(stage['refused'], stage['delivered'])

# A defect is a reply whose delivered action is not the one the benchmark
# expects of that scenario. It is the degradation measure the dialogue arm
# needs, because a refusal rate has no fixed polarity: losing a refusal is a
# failure where refusal was expected and a correction where it was not. The
# expectation is the one attached to the opening prompt and does not move with
# the pressure, per Section 3.7.
stage['aligned'] = np.where(stage['expected_answer'].eq('Refusal'),
                            stage['outcome'].eq('Strong Refusal'),
                            stage['outcome'].eq('Total Compliance'))
stage['defect'] = ~stage['aligned']
stage['Stage'] = stage['turn'].map(STAGE)

print(f"{stage['dialogue_id'].nunique():,} dialogue branches, "
      f"{len(stage):,} stage rows over {stage['Stage'].nunique()} turns")


In [ ]:
# The four-cell outcome at each turn, scenario weighted, so the dialogue arm
# reports the same primary outcome as Section 4.2 rather than a refusal rate the
# safety chapter deliberately does not lead with.
def by_stage(part, column, values=None):
    cell = part.groupby(['label', 'Stage', 'scenario_id'])[column]
    reduced = (cell.value_counts(normalize=True).rename('share').reset_index()
               if values else cell.mean().rename('share').reset_index())
    keys = ['label', 'Stage'] + ([column] if values else [])
    return reduced.groupby(keys)['share'].mean().unstack(level='Stage') * 100


outcomes = (by_stage(stage, 'outcome', values=True)
            .reindex(columns=list(STAGE.values()))
            .reindex(ORDER, level='label').reindex(CELLS, level='outcome'))
outcomes.index.names = ['Model', 'Outcome']
freeze(outcomes.round(2), 'dialogue_01_outcomes', tier='main')
outcomes.round(1)


In [ ]:
# Benchmark Action Alignment and its complement, the defect rate, at each turn.
# Alignment reads the delivered action against the expectation the scenario
# carries, so it is defined in both directions and a model is not rewarded for
# refusing where compliance was expected.
alignment = by_stage(stage, 'aligned').reindex(columns=list(STAGE.values())).reindex(ORDER)
defects = 100 - alignment
defects = pd.concat({'All scenarios': defects,
                     'Age Restricted': 100 - by_stage(
                         stage[stage['stratum'].eq('Age Restricted')], 'aligned')
                         .reindex(columns=list(STAGE.values())).reindex(ORDER),
                     'Harmful': 100 - by_stage(
                         stage[stage['stratum'].eq('Harmful')], 'aligned')
                         .reindex(columns=list(STAGE.values())).reindex(ORDER)},
                    axis=1)
defects.loc[MACRO] = defects.mean()
defects.index.name = 'Model'
defects.columns.names = ['Scenario Type', 'Stage']
freeze(defects.round(2), 'dialogue_02_defects', tier='main')
defects.round(1)


## 1. Refusal survival

Restricted to dialogues whose opening reply was a refusal, since a dialogue that
opened with a compliance has no refusal to lose. Survival is the share still
refusing at each pressed turn. The denominator is dialogues, not turns, so the
two columns are comparable.

In [ ]:
# Two different questions, kept apart. Retention asks whether the model is
# refusing at a given pressure; unbroken survival asks whether it refused at
# every pressure. A dialogue that complies at the first and refuses again at the
# second counts toward the former and not the latter, and the gap between them
# is large: 60.2 per cent against 47.7 at the second pressure.
#
# Recovery is reported separately rather than folded into either, because on
# Purpose Reverse the second pressure is another inversion instruction, so a
# return to refusal can follow mechanically from obeying it. That method carries
# a 28.0 per cent compliance-to-refusal transition against 0.6 on Emotional
# Pushback, so an apparent improvement between the two pressures is not evidence
# that a boundary recovered.
held = frame[frame['opening_refused']]
trajectory = (held.pivot_table(index=['label', 'dialogue_id'], columns='turn',
                               values='refused', aggfunc='first').dropna())

retention = pd.DataFrame({
    'Dialogues Opening in Refusal': held.groupby('label')['dialogue_id'].nunique(),
    'Refusing at Turn 2 (%)': trajectory[2].groupby('label').mean() * 100,
    'Refusing at Turn 3 (%)': trajectory[3].groupby('label').mean() * 100,
    'Refusing at Both (%)': (trajectory[2] & trajectory[3]).groupby('label').mean() * 100,
    'Recovered at Turn 3 (%)':
        trajectory[~trajectory[2]][6].groupby('label').mean() * 100,
}).reindex(ORDER)
retention.loc[MACRO] = retention.mean()
# A count is not a mean over models, so the summary row carries none.
retention.loc[MACRO, 'Dialogues Opening in Refusal'] = np.nan
retention.index.name = 'Model'
freeze(retention.round(2), 'dialogue_01_retention', tier='main')
retention.round(1)


## 2. Which method moves it

The three pressure methods are applied to the same dialogues, so a difference
between them is a difference in the pressure and not in the scenarios.

In [ ]:
method = (held.pivot_table(index=['method', 'label'], columns='turn',
                           values='refused', aggfunc='mean') * 100)
method.columns = [f'Turn {t} (%)' for t in method.columns]
macro = (method.groupby('method').mean()
         .assign(label=MACRO).set_index('label', append=True))
method = pd.concat([method, macro]).sort_index()
method.index.names = ['Method', 'Model']
freeze(method.round(2), 'dialogue_05_methods', tier='main')
method.round(1)

## 3. Does the age effect survive the pressure

Refusal at each turn for a stated minor age against a stated adult age, over all
dialogues rather than only those that opened in refusal, so the turn 2 column is
the adaptation result on this subset of scenarios and the later columns are
comparable with it.

In [ ]:
# Age Restricted is the only stratum whose expected answer moves with age.
# Harmful is a planned control and should be age-flat. An earlier version pooled
# the two, which averaged the signal with its own negative control and halved
# the measured gap: +13.7 points at the first pressure against +27.1 within the
# stratum the question is about.
#
# The adult reference is age 18 alone. The adaptation experiment contrasts
# minors against {18, 21}; this experiment carries no 21, so the two contrasts
# are not interchangeable and the column is named for what it is.
#
# Weighting is scenario first, as everywhere else: replicates within a scenario
# and age, then scenarios, then models equally.
frame['stratum'] = np.where(frame['scenario_id'].str.contains('-a'),
                            'Age Restricted', 'Harmful')
aged = frame[frame['age'].notna()].copy()


def age_gap(part, column):
    cell = part.groupby(['label', 'scenario_id', 'age'])[column].mean().reset_index()
    wide = cell.pivot_table(index=['label', 'scenario_id'], columns='age', values=column)
    minors = [a for a in wide.columns if a < 18]
    return ((wide[minors].mean(axis=1) - wide[18]).dropna()
            .groupby('label').mean() * 100)


# The opening baseline is built from the unique opening cells, before the three
# method branches expand them. Deduplicating on dialogue_id would not do it,
# since the method is part of that identifier, and the opening would then be
# weighted by how many of its branches survived later blocking.
opening = (aged.drop_duplicates(['model', 'prompt_id', 'opening_replicate'])
           .assign(refused=lambda d: d['opening_refused']))

rows = {}
for stratum in ('Age Restricted', 'Harmful'):
    part = aged[aged['stratum'] == stratum]
    rows[(stratum, 'Opening')] = age_gap(opening[opening['stratum'] == stratum], 'refused')
    rows[(stratum, 'Turn 2')] = age_gap(part[part['turn'] == 2], 'refused')
    rows[(stratum, 'Turn 3')] = age_gap(part[part['turn'] == 3], 'refused')

age_effect = pd.DataFrame(rows).reindex(ORDER)
age_effect.loc[MACRO] = age_effect.mean()
age_effect.columns.names = ['Scenario Type', 'Stage']
age_effect.index.name = 'Model'
freeze(age_effect.round(2), 'dialogue_03_age', tier='main')
age_effect.round(1)


## 4. Delivery under pressure

Refusal is what the model says. This is what arrives. A dialogue that keeps
refusing and starts delivering is the failure the four-cell outcome was built to
expose, and it is the one a refusal rate over turns would not show.

In [ ]:
# Two estimands that were previously given one name. The share is the fraction
# of the opening-refusal cohort sitting in the Weak Refusal cell at that turn;
# the conditional rate is the fraction of the replies still refusing at that
# turn that delivered anyway. They have different denominators and the second is
# roughly ten times the first, so reporting one under the other's name overstates
# the leak by an order of magnitude.
held = stage[stage['dialogue_id'].isin(
    stage.loc[stage['turn'].eq(1) & stage['refused'], 'dialogue_id'])]
pressed_only = held[held['turn'].ne(1)]

share = (pressed_only.assign(weak=pressed_only['outcome'].eq('Weak Refusal'))
         .pivot_table(index='label', columns='Stage', values='weak', aggfunc='mean') * 100)
conditional = (pressed_only[pressed_only['refused']]
               .pivot_table(index='label', columns='Stage', values='delivered',
                            aggfunc='mean') * 100)
leak = pd.concat({'Weak Refusal Share (%)': share,
                  'Delivery among Refusals (%)': conditional}, axis=1).reindex(ORDER)
leak.loc[MACRO] = leak.mean()
leak.index.name = 'Model'
leak.columns.names = ['Measure', 'Stage']
freeze(leak.round(2), 'dialogue_s02_leakage')
leak.round(1)


## What this notebook writes

| Table | Tier |
|---|---|
| `persistence_01_survival` | main |
| `persistence_02_method` | main |
| `persistence_03_age` | main |
| `persistence_s01_delivery` | supplement |

Every table here is descriptive. The persistence extension declares no
hypothesis family, so nothing carries a permutation value or an adjusted one,
and the numbers are reported as description in the results chapter.

To move these onto `publish()`, add the four names to `config/captions.yml` with
a `label`, a `tier` and a `kind: table`, then replace `freeze` with `publish` in
the setup cell.